# CoolProp Backend Test

Functional tests for the optional CoolProp backend added in v0.4.4.

The notebook checks:
- optional import behaviour,
- pure-fluid property retrieval,
- comparison of CoolProp water properties with the IAPWS provider,
- CoolProp mixture-string generation,
- gas-mixture property retrieval where supported by CoolProp.

Core units:
- temperature: K
- pressure: Pa
- density: kg/m3
- dynamic viscosity: Pa*s
- thermal conductivity: W/(m*K)
- specific heat: J/(kg*K)
- specific enthalpy: J/kg


In [ ]:
from pathlib import Path
import sys
import math

import pandas as pd

# Make the notebook usable both from repository root and from core/tests.
cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]

for candidate in candidate_roots:
    if (candidate / "core").is_dir():
        workspace_root = candidate
        break
else:
    raise RuntimeError("Could not find repository root containing the 'core' directory.")

if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

print("Workspace root:", workspace_root)


In [ ]:
try:
    import CoolProp.CoolProp as CP
    COOLPROP_AVAILABLE = True
except ImportError:
    COOLPROP_AVAILABLE = False

from core.properties import (
    CoolPropFluidProvider,
    CoolPropGasMixtureProvider,
    build_coolprop_mixture_string,
    coolprop_props,
    normalize_mole_fractions,
    to_internal_fluid_props,
    to_outside_fluid_props,
    water_steam_props_iapws97,
)

if not COOLPROP_AVAILABLE:
    print("CoolProp is not installed. Install optional dependency with: pip install CoolProp")
else:
    air = coolprop_props(T=293.15, p=101325.0, fluid="Air")
    assert 1.15 < air.transport.rho < 1.25, air.transport.rho
    assert 1.7e-5 < air.transport.mu < 1.9e-5, air.transport.mu
    assert 0.024 < air.transport.k < 0.027, air.transport.k
    assert 990.0 < air.transport.cp < 1020.0, air.transport.cp

    provider = CoolPropFluidProvider("Air")
    provider_props = provider.at(T=293.15, p=101325.0)
    assert provider_props.rho == air.transport.rho

    print("CoolProp imports and smoke test passed.")


In [ ]:
if COOLPROP_AVAILABLE:
    pure_fluid_points = [
        {"case": "air ambient", "fluid": "Air", "T_C": 20.0, "p_bar": 1.01325},
        {"case": "air hot gas", "fluid": "Air", "T_C": 180.0, "p_bar": 1.01325},
        {"case": "carbon dioxide ambient", "fluid": "CO2", "T_C": 20.0, "p_bar": 1.01325},
        {"case": "water ambient", "fluid": "Water", "T_C": 20.0, "p_bar": 1.01325},
        {"case": "water pressurized hot", "fluid": "Water", "T_C": 120.0, "p_bar": 3.0},
    ]

    rows = []

    for point in pure_fluid_points:
        T = point["T_C"] + 273.15
        p = point["p_bar"] * 1.0e5

        try:
            result = coolprop_props(T=T, p=p, fluid=point["fluid"])
            warnings = ", ".join(w.code for w in result.warnings)

            rows.append({
                "case": point["case"],
                "fluid": point["fluid"],
                "T_C": point["T_C"],
                "p_bar": point["p_bar"],
                "phase": result.phase,
                "rho_kg_m3": result.transport.rho,
                "mu_Pa_s": result.transport.mu,
                "k_W_mK": result.transport.k,
                "cp_J_kgK": result.transport.cp,
                "h_kJ_kg": result.h / 1000.0,
                "warnings": warnings,
                "error": "",
            })

        except Exception as exc:
            rows.append({
                "case": point["case"],
                "fluid": point["fluid"],
                "T_C": point["T_C"],
                "p_bar": point["p_bar"],
                "phase": "",
                "rho_kg_m3": math.nan,
                "mu_Pa_s": math.nan,
                "k_W_mK": math.nan,
                "cp_J_kgK": math.nan,
                "h_kJ_kg": math.nan,
                "warnings": "",
                "error": str(exc),
            })

    coolprop_pure_df = pd.DataFrame(rows)

    air_row = coolprop_pure_df.loc[coolprop_pure_df["case"] == "air ambient"].iloc[0]
    assert air_row["error"] == ""
    assert 1.15 < air_row["rho_kg_m3"] < 1.25
    assert 1.7e-5 < air_row["mu_Pa_s"] < 1.9e-5

    water_row = coolprop_pure_df.loc[coolprop_pure_df["case"] == "water ambient"].iloc[0]
    iapws_water = water_steam_props_iapws97(T=293.15, p=101325.0)

    assert water_row["error"] == ""
    assert abs(water_row["rho_kg_m3"] - iapws_water.transport.rho) / iapws_water.transport.rho < 0.01

    print("CoolProp pure-fluid matrix test passed.")
    display(coolprop_pure_df)
else:
    print("Skipped pure-fluid matrix test because CoolProp is not installed.")

In [ ]:
if COOLPROP_AVAILABLE:
    mixture_cases = [
        {
            "case": "dry flue gas candidate",
            "components": {
                "Nitrogen": 0.74,
                "Oxygen": 0.04,
                "CarbonDioxide": 0.12,
            },
            "T_C": 180.0,
            "p_bar": 1.01325,
        },
        {
            "case": "wet flue gas candidate",
            "components": {
                "Nitrogen": 0.70,
                "Oxygen": 0.04,
                "CarbonDioxide": 0.12,
                "Water": 0.14,
            },
            "T_C": 180.0,
            "p_bar": 1.01325,
        },
    ]

    rows = []

    for case in mixture_cases:
        T = case["T_C"] + 273.15
        p = case["p_bar"] * 1.0e5
        fluid = build_coolprop_mixture_string(case["components"])

        try:
            provider = CoolPropGasMixtureProvider(case["components"])
            result = provider.full_at(T=T, p=p)

            rows.append({
                "case": case["case"],
                "T_C": case["T_C"],
                "p_bar": case["p_bar"],
                "fluid": fluid,
                "phase": result.phase,
                "rho_kg_m3": result.transport.rho,
                "mu_Pa_s": result.transport.mu,
                "k_W_mK": result.transport.k,
                "cp_J_kgK": result.transport.cp,
                "h_kJ_kg": result.h / 1000.0,
                "warnings": ", ".join(w.code for w in result.warnings),
                "error": "",
            })

        except Exception as exc:
            rows.append({
                "case": case["case"],
                "T_C": case["T_C"],
                "p_bar": case["p_bar"],
                "fluid": fluid,
                "phase": "",
                "rho_kg_m3": math.nan,
                "mu_Pa_s": math.nan,
                "k_W_mK": math.nan,
                "cp_J_kgK": math.nan,
                "h_kJ_kg": math.nan,
                "warnings": "",
                "error": str(exc),
            })

    coolprop_mixture_df = pd.DataFrame(rows)

    # Mixture support depends on CoolProp backend data availability. The test is considered
    # useful if it either returns properties or provides a readable backend error per case.
    assert len(coolprop_mixture_df) == len(mixture_cases)

    print("CoolProp mixture matrix evaluated. Inspect errors if any mixture is unsupported by CoolProp.")
    display(coolprop_mixture_df)
else:
    print("Skipped mixture matrix test because CoolProp is not installed.")

In [ ]:
if COOLPROP_AVAILABLE:
    provider = CoolPropFluidProvider("Air")
    props = provider.at(T=293.15, p=101325.0)

    internal_props = to_internal_fluid_props(props)
    outside_props = to_outside_fluid_props(props)

    assert internal_props.rho == props.rho
    assert internal_props.mu == props.mu
    assert internal_props.k == props.k
    assert internal_props.cp == props.cp

    assert outside_props.rho == props.rho
    assert outside_props.mu == props.mu
    assert outside_props.k == props.k
    assert outside_props.cp == props.cp

    print("CoolProp provider adapter test passed.")
else:
    print("Skipped provider adapter test because CoolProp is not installed.")